# Random Forest Volatility Backtest

CPU-only tree-based walk-forward backtest. sklearn's RandomForestRegressor
handles missing values via surrogate splits, so no scaling or imputation
needed.

**Architecture (post Step 9 dedup).** This notebook now contains only
RandomForest-specific glue:
* `DEFAULT_RF_PARAMS` — method defaults merged with `--params-file`.
* `fit_predict_rf` — closure over `run_backtest(use_scaling=False)`.
* `main()` — three-line wrapper around `parse_executor_args` +
  `run_executor`.

All dataset loading, transforms, HAR/calendar features, horizon shift,
chunk slicing, smearing, and reduce-JSON writing live in
`src/executor.py` (exported from `notebooks/pipeline/05_executor.ipynb`).

**Behavior change vs pre-migration.** The pre-migration executor wrote
a custom `metrics_chunk_*.json` per chunk; the migrated path writes the
standard `_reduce.json` produced by `src.evaluation.save_chunk_reduce`
(same format as XGB/LGBM/Ridge use). Existing trial outputs remain
valid; only future runs are affected.

In [ ]:
# ── Setup: clone repo, install deps ──
import os

REPO_URL = "https://github.com/jamesdchen/harxhar.git"
REPO_DIR = "/content/harxhar"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -q numpy pandas scikit-learn pyarrow tqdm

import sys  # noqa: E402

sys.path.insert(0, "/content/harxhar")

In [ ]:
# ── Notebook-only demo parameters ──
HORIZON = 1
TRAIN_WINDOW_DAYS = 500
REFIT_FREQUENCY = 5
DATA_PATH = "all30min"

## Module imports + defaults

First export cell: docstring, imports, default-hyperparam dict.

In [ ]:
# export
"""Random Forest volatility backtest executor.

Method-specific glue around the shared scaffold in :mod:`src.executor`.
Only the model factory + default hyperparams + ``main()`` wrapper live
here; everything else (CLI parsing, loading, features, backtest loop,
smearing, reduce JSON) is owned by ``src.executor``.
"""

from __future__ import annotations

import json

import numpy as np
from sklearn.ensemble import RandomForestRegressor

from src.executor import CONFIGS, run_executor
from src.loading import parse_exog_cols
from src.scaling import run_backtest

# Per-method default hyperparams. Tuned overrides from --params-file are
# merged on top via dict.update().
DEFAULT_RF_PARAMS: dict = dict(
    n_estimators=500,
    max_depth=10,
    min_samples_leaf=5,
    n_jobs=-1,
)


In [ ]:
# ── Imports smoke test ──
_expected = [
    "json",
    "np",
    "RandomForestRegressor",
    "run_executor",
    "parse_exog_cols",
    "run_backtest",
    "DEFAULT_RF_PARAMS",
]
_missing = [n for n in _expected if n not in dir()]
assert not _missing, f"missing imports: {_missing}"
print(f"imports OK ({len(_expected)} symbols)")
print(f"DEFAULT_RF_PARAMS={DEFAULT_RF_PARAMS}")

## `fit_predict_rf` — the only model-specific call

Closure over `run_backtest` with `use_scaling=False` (RandomForest does
not need feature standardization). Each call builds a fresh
model-factory bound to the merged hyperparams, then runs the shared
walk-forward loop.

**`random_state=42` is wired here** (via `hyperparams['random_state']`,
default 42 from `parse_executor_args --seed`) so every refit inside
`run_backtest` uses the same seed for reproducibility.

In [ ]:
# export
def fit_predict_rf(
    X_chunk: np.ndarray,
    y_chunk: np.ndarray,
    train_win_periods: int,
    hyperparams: dict,
) -> np.ndarray:
    """Walk-forward backtest with RandomForest. Returns OOS predictions.

    Wraps :func:`src.scaling.run_backtest` with the RF-specific settings
    (``use_scaling=False``; refit frequency from
    ``hyperparams['_refit_frequency']``). ``random_state`` defaults to
    42 and can be overridden via ``hyperparams['random_state']``.
    """
    refit_frequency = int(hyperparams.get("_refit_frequency", CONFIGS["random_forest"].refit_frequency))
    # Strip our internal control key before passing to RandomForestRegressor.
    model_kwargs = {k: v for k, v in hyperparams.items() if not k.startswith("_")}
    model_kwargs.setdefault("random_state", 42)

    def model_fn():
        return RandomForestRegressor(**model_kwargs)

    return run_backtest(
        model_fn,
        X_chunk,
        y_chunk,
        train_win=train_win_periods,
        refit_frequency=refit_frequency,
        use_scaling=False,
    )


In [ ]:
# ── Smoke-test fit_predict_rf on synthetic data ──
rng = np.random.default_rng(42)
n, k = 600, 4
X_demo = rng.normal(size=(n, k))
y_demo = rng.normal(size=n)
_hp = dict(DEFAULT_RF_PARAMS, _refit_frequency=10, n_estimators=20, random_state=42)  # tiny for speed
preds = fit_predict_rf(X_demo, y_demo, train_win_periods=500, hyperparams=_hp)
assert preds.shape == (n - 500,), preds.shape
assert np.all(np.isfinite(preds)), "predictions must be finite"
print(f"fit_predict_rf OK: preds.shape={preds.shape}, mean={preds.mean():.4f}")

## In-notebook demo (manual pipeline)

A didactic walk-through of the full pipeline using the in-notebook
constants above. The exported `main()` re-implements this via
`run_executor`; this section is here so a reader can see each step.

In [ ]:
# ── Load + transform ──
import numpy as np

from src.loading import load_raw_data
from src.transforms import robust_transform

df = load_raw_data(DATA_PATH, allow_missing=True)
print(f"Loaded: {df.shape}")

# Full transform on RV target (diurnal + winsor)
adj_rv, baseline = robust_transform(df, "RV", is_target=True, use_diurnal=True, winsor_window=240)
df["adj_RV"] = adj_rv
df["baseline"] = baseline

print(f"adj_RV stats:\n{df['adj_RV'].describe()}")

In [ ]:
# --- Features from shared pipeline ---
from src.transforms import resolve_har_lags, generate_har_features, add_calendar_features

df, har_names = generate_har_features(df, target_col="adj_RV")
cal_names = add_calendar_features(df)
feature_names = har_names + cal_names

print(f"HAR lags: {resolve_har_lags()}")
print(f"Features ({len(feature_names)}): {feature_names}")

# Drop initial NaN rows from HAR computation
max_lag = resolve_har_lags()[-1]
df = df.iloc[max_lag:].reset_index(drop=True)
print(f"Shape after lag trim: {df.shape}")

In [ ]:
# --- Random Forest walk-forward backtest (demo) ---
from sklearn.ensemble import RandomForestRegressor
from src.transforms import apply_horizon_shift
from src.scaling import run_backtest
from src.transforms import PERIODS_PER_DAY

X = df[feature_names].values.astype(np.float64)
y = df["adj_RV"].values.astype(np.float64)
dates = df["t"]
baselines = df["baseline"].values.astype(np.float64)

# Horizon shift
X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, HORIZON)

train_win = TRAIN_WINDOW_DAYS * PERIODS_PER_DAY
print(f"Samples: {len(y)}, Features: {X.shape[1]}, Train window: {train_win}")

model_fn = lambda: RandomForestRegressor(**DEFAULT_RF_PARAMS, random_state=42)

predictions = run_backtest(model_fn, X, y, train_win=train_win, refit_frequency=REFIT_FREQUENCY, use_scaling=False)
print(f"Predictions: {predictions.shape}, NaN count: {np.isnan(predictions).sum()}")

# Fit final model on last window for feature importance plot
model = model_fn()
model.fit(X[-train_win:], y[-train_win:])

In [ ]:
# ── Feature importance ──
import matplotlib.pyplot as plt

importance = model.feature_importances_
sorted_idx = np.argsort(importance)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([feature_names[i] for i in sorted_idx], importance[sorted_idx])
ax.set_xlabel("Feature Importance (impurity decrease)")
ax.set_title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()

In [ ]:
# --- Evaluation ---
from src.evaluation import apply_duan_smearing, calculate_metrics
import pandas as pd

oos_start = train_win
y_oos = y[oos_start:]
dates_oos = dates.iloc[oos_start:].values
baselines_oos = baselines[oos_start:]

pred_raw, true_raw = apply_duan_smearing(predictions, y_oos, baselines_oos)

results_df = pd.DataFrame(
    {
        "date": dates_oos,
        "horizon": HORIZON,
        "true_adj": y_oos,
        "pred_adj": predictions,
        "true_raw": true_raw,
        "pred_raw": pred_raw,
    }
)

metrics = calculate_metrics(results_df)
print("\n".join(f"{k:>10s}: {v:.6f}" for k, v in metrics.items()))

## `main()` — CLI entry point

Three responsibilities:
1. Parse the canonical CLI via `parse_executor_args`.
2. Merge `--params-file` JSON onto `DEFAULT_RF_PARAMS`; bake
   `random_state=args.seed` into the merged hyperparams so it flows
   through to every refit inside `run_backtest`.
3. Hand off to `run_executor` with method-specific toggles
   (`add_calendar=True`, `target_use_diurnal=True`,
   `target_winsor_window=240`, `dropna_with_exog=True` — RF historically
   uses intersection-N dropna for exog runs).

The `--segment` / `--lag-scope` flags are present in the parser for
tuner-CLI uniformity but are passed through unused (RandomForest runs
only in segment-less / `lag_scope='global'` mode).

In [ ]:
# export
def compute(args) -> None:
    tuned_params: dict = {}
    if args.params_file:
        with open(args.params_file) as f:
            tuned_params = json.load(f)

    # Method defaults <- tuned overrides; refit-frequency from CLI sentinel.
    hyperparams = dict(DEFAULT_RF_PARAMS)
    hyperparams.update(tuned_params)
    # Wire seed through to RandomForestRegressor's random_state.
    hyperparams.setdefault("random_state", args.seed)
    refit_frequency = (
        args.refit_frequency
        if args.refit_frequency is not None
        else CONFIGS["random_forest"].refit_frequency
    )
    hyperparams["_refit_frequency"] = refit_frequency

    exog_cols = parse_exog_cols(args.exog_cols)

    run_executor(
        method_name="random_forest",
        fit_predict=fit_predict_rf,
        hyperparams=hyperparams,
        data_path=args.data_path,
        output_file=args.output_file,
        horizon=args.horizon,
        train_window=args.train_window,
        start=args.start,
        end=args.end,
        exog_cols=exog_cols,
        segment=args.segment,
        lag_scope=args.lag_scope,
        **CONFIGS["random_forest"].as_data_prep_kwargs(),
        seed=args.seed,
    )


In [ ]:
# ── Smoke-test main() signature ──
import inspect

assert callable(main)
sig = inspect.signature(main)
assert list(sig.parameters) == [], f"main() should take no args; got {sig}"
print("main signature OK:", sig)

In [ ]:
import sys; sys.path.insert(0, "..")
from _exporter import export_notebook

export_notebook("ml_random_forest.ipynb", "../../src/ml_random_forest.py")